# FilingIQ — Chunking Configuration Evaluation

This is a **local Jupyter notebook** (not a Snowflake Notebook). It talks
to Snowflake over `snowflake-connector-python` and generates answers via
Groq's free API, matching `app/streamlit_app.py`.

**Why local, not a Snowflake Notebook:** on this project's trial account,
every Cortex/AI function (`SNOWFLAKE.CORTEX.COMPLETE`, `AI_COMPLETE`,
`EMBED_TEXT_768`) is blocked — see the note at the top of
`sql/03_process_procedure.sql` for the exact errors. Parsing/chunking
were reworked around that with plain Python UDFs (`PDF_TO_TEXT`,
`CHUNK_TEXT`), but there's still no in-Snowflake way to call an LLM on a
trial account, so this notebook uses Groq instead. Snowflake Notebooks
(like Streamlit in Snowflake) run in a sandboxed container that can't
make arbitrary outbound HTTPS calls to Groq without extra network/
External Access Integration setup — running this locally sidesteps that
entirely.

**What this notebook does:**
1. Builds 3 chunk-table variants (`DOC_CHUNKS_1000_200`,
   `DOC_CHUNKS_1800_300`, `DOC_CHUNKS_3000_500`) by re-running the
   `CHUNK_TEXT` UDF against the already-parsed text in `PARSED_DOCS` with
   three different size/overlap settings — no PDF re-parsing needed.
2. Runs the full RAG pipeline (TF-IDF retrieve -> grounded prompt ->
   Groq) for every question in `questions.csv`, against every config.
3. Scores every answer with **TruLens**, using its Snowflake connector to
   persist feedback results as native Snowflake tables, and a LiteLLM-
   backed feedback provider pointed at Groq (TruLens's Cortex provider
   isn't usable here for the same reason Cortex COMPLETE isn't).
4. Also computes LLM-as-judge **accuracy** against your `expected_answer`
   column and **refusal rate** on the unanswerable questions.
5. Saves results to `eval/results/` and prints the recommended config.

**Before running:**
- `sql/01_setup.sql` through `sql/05_search_service.sql` must already be
  applied, and `PARSED_DOCS` / `DOC_CHUNKS` already populated (this
  notebook reuses `PARSED_DOCS`, it does not re-parse PDFs).
- Fill in `expected_answer` in `eval/questions.csv` first (see
  `eval/QUESTION_GUIDE.md`).
- From the `eval/` folder: `pip install -r requirements.txt`, then
  `cp .env.example .env` and fill it in (same shape as `app/.env.example`
  plus `GROQ_API_KEY`).
- VERIFY: TruLens's package names/import paths
  (`trulens-core`, `trulens-providers-litellm`, `trulens-connectors-snowflake`)
  and the `SnowflakeConnector`/`LiteLLM` constructor kwargs used below —
  TruLens's packaging has changed across releases; check
  `pypi.org/project/trulens-connectors-snowflake` if an import fails.


In [ ]:
import os
import time
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector
from dotenv import load_dotenv

load_dotenv()

CONN_PARAMS = dict(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    role=os.getenv("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE", "FILINGIQ_WH"),
    database="FILINGIQ_DB",
    schema="PUBLIC",
)
_password = os.getenv("SNOWFLAKE_PASSWORD")
if _password:
    CONN_PARAMS["password"] = _password
else:
    CONN_PARAMS["authenticator"] = "externalbrowser"

GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"
ANSWER_MODEL = "openai/gpt-oss-20b"  # switched from 120b: live-tested 2026-11-11, 120b hits an 8000 TPM wall
                                      # almost immediately under this evaluation's call volume (90
                                      # question x config iterations x up to 4 Groq calls each), while 20b
                                      # has separate headroom and is faster -- worth it for a batch eval
                                      # run even though the live app defaults to 120b for quality.

conn = snowflake.connector.connect(**CONN_PARAMS)

def run_query(sql: str, params=None) -> pd.DataFrame:
    cur = conn.cursor()
    try:
        cur.execute(sql, params)
        return cur.fetch_pandas_all()
    finally:
        cur.close()

def run_sql(sql: str, params=None):
    cur = conn.cursor()
    try:
        cur.execute(sql, params)
    finally:
        cur.close()

os.makedirs("eval/results", exist_ok=True)
print("Connected:", run_query("SELECT CURRENT_WAREHOUSE() AS wh, CURRENT_DATABASE() AS db").to_dict("records"))


## 1. Build the three chunking-configuration variants

Each variant gets its own `DOC_CHUNKS_<suffix>` table, built by re-running
the `CHUNK_TEXT` UDF (defined in `sql/03_process_procedure.sql`) against
`PARSED_DOCS` with different size/overlap settings. No new Snowflake
object (like a Cortex Search Service) is needed per variant here — search
against a variant table is done with inline TF-IDF SQL in section 3,
parameterized by table name.

In [ ]:
def build_chunk_variant(suffix: str, chunk_size: int, overlap: int) -> str:
    """Re-chunk every row of PARSED_DOCS into DOC_CHUNKS_<suffix>, using the
    same CHUNK_TEXT -> filename-prefix -> SHA2-dedup logic as
    PROCESS_NEW_DOCS() in sql/03_process_procedure.sql."""
    table = f"DOC_CHUNKS_{suffix}"

    run_sql(f'''
        CREATE OR REPLACE TABLE {table} (
            chunk_id       STRING DEFAULT UUID_STRING(),
            relative_path  STRING,
            company        STRING,
            fiscal_year    NUMBER,
            file_url       STRING,
            chunk_index    NUMBER,
            chunk          STRING,
            chunk_hash     STRING
        )
    ''')

    run_sql(f'''
        INSERT INTO {table} (relative_path, company, fiscal_year, file_url, chunk_index, chunk, chunk_hash)
        SELECT
            p.relative_path,
            p.company,
            p.fiscal_year,
            p.file_url,
            f.index,
            SPLIT_PART(p.relative_path, '/', -1) || ': ' || f.value::STRING AS chunk_text,
            SHA2(SPLIT_PART(p.relative_path, '/', -1) || ': ' || f.value::STRING)
        FROM PARSED_DOCS p,
             TABLE(FLATTEN(INPUT => CHUNK_TEXT(p.parsed_text, {chunk_size}, {overlap}))) f
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY p.relative_path, SHA2(SPLIT_PART(p.relative_path, '/', -1) || ': ' || f.value::STRING)
            ORDER BY f.index
        ) = 1
    ''')

    count = run_query(f"SELECT COUNT(*) AS c FROM {table}")["C"].iloc[0]
    print(f"Built {table}: {count} chunks.")
    return table


CONFIGS = {
    "1000_200": (1000, 200),
    "1800_300": (1800, 300),
    "3000_500": (3000, 500),
}

variant_tables = {suffix: build_chunk_variant(suffix, size, overlap) for suffix, (size, overlap) in CONFIGS.items()}
variant_tables


## 2. RAG pipeline helpers

Same TF-IDF-search + grounded-prompt + Groq logic as
`app/streamlit_app.py`, parameterized so it can run against any of the
three chunk-table variants (or the production `DOC_CHUNKS` table).

In [ ]:
def search_variant(table: str, query: str, company, fiscal_year, k: int = 5) -> pd.DataFrame:
    """Inline version of the SEARCH_CHUNKS TF-IDF logic from
    sql/05_search_service.sql, parameterized by table name. `table` is
    always one of our own fixed CONFIGS/production table names (never
    user input), so building it into the SQL text is safe; query/company/
    fiscal_year/k are passed as bind parameters."""
    # Matches the fixed sql/05_search_service.sql SEARCH_CHUNKS logic:
    # strip the filename prefix before scoring, drop stopwords, and
    # unconditionally exclude known company names/tickers (found live:
    # without this, two different questions that both happen to mention
    # a company name can score the exact same chunks identically, since
    # the company name ends up the only term that matches anything).
    sql = f'''
        WITH terms AS (
            SELECT DISTINCT t.value::STRING AS term
            FROM TABLE(SPLIT_TO_TABLE(REGEXP_REPLACE(LOWER(%(query)s), '[^a-z0-9 ]', ' '), ' ')) t
            WHERE LENGTH(t.value) > 2
              AND t.value NOT IN (
                    'the','a','an','of','and','or','to','in','on','for','with',
                    'from','that','this','these','those','are','its','their',
                    'what','which','who','whose','why','when','how',
                    'does','did','do','is','was','were','has','have',
                    'mention','mentions','mentioned','related','about','main'
              )
              AND t.value NOT IN (
                    'aapl','apple','msft','microsoft','nvda','nvidia',
                    'googl','google','amzn','amazon','meta','tsla','tesla',
                    'jpm','jpmorgan','nflx','netflix'
              )
        ),
        cand AS (
            SELECT chunk_id, chunk, company, fiscal_year, relative_path, chunk_index,
                   LOWER(SUBSTR(chunk, POSITION(': ' IN chunk) + 2)) AS lc
            FROM {table}
            WHERE (%(company)s IS NULL OR company = UPPER(%(company)s))
              AND (%(year)s IS NULL OR fiscal_year = %(year)s)
        ),
        n AS (
            SELECT COUNT(*) AS n FROM cand
        ),
        df AS (
            SELECT t.term, COUNT_IF(CONTAINS(c.lc, t.term)) AS df
            FROM terms t CROSS JOIN cand c
            GROUP BY t.term
        ),
        scored AS (
            SELECT c.chunk_id, c.chunk, c.company, c.fiscal_year, c.relative_path, c.chunk_index,
                   SUM(
                       ((LENGTH(c.lc) - LENGTH(REPLACE(c.lc, t.term, ''))) / LENGTH(t.term))
                       * LN(1 + n.n / NULLIF(df.df, 0))
                   ) AS score
            FROM cand c
            CROSS JOIN terms t
            JOIN df ON df.term = t.term
            CROSS JOIN n
            GROUP BY c.chunk_id, c.chunk, c.company, c.fiscal_year, c.relative_path, c.chunk_index
        )
        SELECT chunk, company, fiscal_year, relative_path, chunk_index, score
        FROM scored
        WHERE score > 0
        QUALIFY ROW_NUMBER() OVER (ORDER BY score DESC) <= {int(k)}
    '''
    return run_query(sql, {"query": query, "company": company, "year": fiscal_year})


def groq_chat(model: str, system_prompt: str, user_prompt: str, temperature: float = 0.0) -> str:
    api_key = os.environ["GROQ_API_KEY"]
    resp = requests.post(
        GROQ_URL,
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": model,
            "temperature": temperature,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        },
        timeout=60,
    )
    if resp.status_code == 429:
        time.sleep(20)
        resp = requests.post(
            GROQ_URL,
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": model, "temperature": temperature, "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ]},
            timeout=60,
        )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]


SYSTEM_PROMPT = (
    "You are FilingIQ, an analyst that answers questions about SEC 10-K filings. "
    "Answer ONLY from the numbered excerpts provided. Cite excerpts inline like [1] or [2]. "
    "Quote exact figures (with units and fiscal year) when the excerpts contain them. "
    "If the excerpts do not contain the answer, reply exactly: Not found in the filings."
)


def build_context(chunks: pd.DataFrame) -> str:
    parts = []
    for i, row in enumerate(chunks.itertuples(index=False), start=1):
        parts.append(f"[{i}] ({row.RELATIVE_PATH}, chunk {row.CHUNK_INDEX})\n{row.CHUNK}")
    return "\n\n".join(parts)


def answer_question(table: str, question: str, company: str, fiscal_year_raw, is_unanswerable: bool):
    fy_raw = str(fiscal_year_raw)
    single_year = None if "-" in fy_raw else int(fy_raw)
    filter_company = None if is_unanswerable else company

    chunks = search_variant(table, question, filter_company, single_year, k=5)
    if chunks.empty:
        return "Not found in the filings.", [], ""

    context_text = "\n\n".join(chunks["CHUNK"].tolist())
    answer = groq_chat(ANSWER_MODEL, SYSTEM_PROMPT, f"Excerpts:\n\n{build_context(chunks)}\n\nQuestion: {question}")
    sources = sorted(set(chunks["RELATIVE_PATH"].tolist()))
    return answer, sources, context_text


## 3. TruLens feedback functions (groundedness, context relevance, answer relevance)

Uses TruLens's `SnowflakeConnector` purely to persist feedback-run
metadata as native Snowflake tables, and a `LiteLLM`-backed feedback
provider pointed at Groq to actually compute the scores (TruLens's Cortex
provider needs `SNOWFLAKE.CORTEX.COMPLETE`, which is blocked here).

In [ ]:
# VERIFY: import paths and constructor kwargs below -- TruLens's provider/
# connector packaging has shifted across releases.
from trulens.providers.litellm import LiteLLM
from trulens.connectors.snowflake import SnowflakeConnector
from trulens.core import TruSession

# litellm's "groq/<model>" provider prefix picks up GROQ_API_KEY from the
# environment automatically.
feedback_provider = LiteLLM(model_engine=f"groq/{ANSWER_MODEL}")

tru_connector = SnowflakeConnector(**CONN_PARAMS)
tru_session = TruSession(connector=tru_connector)


import threading

def _call_with_hard_timeout(func, args, kwargs, timeout_seconds):
    """Run func in a daemon thread with a hard wall-clock deadline. A live
    run once sat completely idle (0.0% CPU, 1.8s of actual CPU time) for
    2.5 hours on a single call -- no exception, no progress, just a
    permanently blocked network call somewhere under LiteLLM/requests
    with no timeout of its own. join(timeout=...) is what actually bounds
    that: if the thread hasn't finished by the deadline we give up and
    move on, instead of trusting the call to eventually raise or return.
    daemon=True is deliberate: an abandoned, truly-hung thread must never
    block the rest of this notebook (or the process itself) from exiting."""
    box = {}
    def _target():
        try:
            box["value"] = func(*args, **kwargs)
        except Exception as e:
            box["error"] = e
    t = threading.Thread(target=_target, daemon=True)
    t.start()
    t.join(timeout=timeout_seconds)
    if t.is_alive():
        raise TimeoutError(f"{getattr(func, '__name__', 'call')} exceeded {timeout_seconds}s hard timeout")
    if "error" in box:
        raise box["error"]
    return box.get("value")


def _with_retry(func, *args, max_attempts=2, base_delay=8, call_timeout=75, **kwargs):
    """TruLens/LiteLLM already retries internally several times (its own
    backoff) before raising -- stacking a long retry on top of that
    multiplies the worst case instead of helping (an earlier run sat for
    20+ minutes on a single row because of exactly this). This layer
    exists to (a) bound every call with a hard timeout (see
    _call_with_hard_timeout above) so nothing can ever hang indefinitely
    again, (b) make one extra attempt with a short, non-multiplying
    delay, and (c) guarantee a failure degrades to a missing (NaN) metric
    instead of crashing the whole 90-iteration loop."""
    for attempt in range(max_attempts):
        try:
            return _call_with_hard_timeout(func, args, kwargs, call_timeout)
        except TimeoutError as e:
            print(f"  [warn] {e}")
            if attempt == max_attempts - 1:
                return None
            time.sleep(base_delay)
        except Exception as e:
            if attempt == max_attempts - 1:
                print(f"  [warn] {func.__name__} failed after {max_attempts} attempts: {e}")
                return None
            print(f"  [retry] {func.__name__} hit an error ({type(e).__name__}), waiting {base_delay}s...")
            time.sleep(base_delay)


def groundedness_score(context_text: str, answer_text: str) -> float:
    if not context_text.strip():
        return 0.0
    score, _ = feedback_provider.groundedness_measure_with_cot_reasons(source=context_text, statement=answer_text)
    return float(score)


def context_relevance_score(question: str, context_text: str) -> float:
    if not context_text.strip():
        return 0.0
    score, _ = feedback_provider.context_relevance_with_cot_reasons(question=question, context=context_text)
    return float(score)


def answer_relevance_score(question: str, answer_text: str) -> float:
    score, _ = feedback_provider.relevance_with_cot_reasons(prompt=question, response=answer_text)
    return float(score)


def judge_accuracy(question: str, expected_answer, actual_answer: str):
    """LLM-as-judge via Groq: does the generated answer convey the same key
    facts as the human-filled expected_answer? None when expected_answer
    is blank (e.g. unanswerable rows -- nothing to grade against)."""
    if expected_answer is None or (isinstance(expected_answer, float) and pd.isna(expected_answer)) or str(expected_answer).strip() == "":
        return None
    prompt = f'''You are grading an AI-generated answer against a reference
answer for a question about an SEC 10-K filing.

Question: {question}
Reference answer: {expected_answer}
AI answer: {actual_answer}

Does the AI answer convey the same key facts as the reference answer
(wording may differ, but numbers/names/facts must match)? Respond with
ONLY a single digit: 1 for correct/equivalent, 0 for incorrect or missing
key facts.'''
    verdict = groq_chat(ANSWER_MODEL, "You are a strict grading assistant.", prompt).strip()
    digits = [ch for ch in verdict if ch in "01"]
    return int(digits[0]) if digits else 0


def is_refusal(answer_text: str) -> bool:
    return "not found in the filings" in answer_text.lower()


## 4. Load questions and run the full evaluation grid

Runs all 30 questions against all 3 chunking configs. Groq's free tier is
rate-limited, so this includes a small delay between calls and a one-time
retry on HTTP 429 (see `groq_chat` above) — expect this cell to take
several minutes.

In [ ]:
questions_df = pd.read_csv("questions.csv")
assert {"question", "expected_answer", "company", "fiscal_year", "type"}.issubset(questions_df.columns)
print(f"Loaded {len(questions_df)} questions.")
questions_df.head()


In [ ]:
all_rows = []
total = len(variant_tables) * len(questions_df)
done = 0

# Pacing: this account's Groq tier caps at 8000 tokens/minute for
# openai/gpt-oss-120b. A live run without inter-call spacing burned
# through that in well under a minute and crashed with zero results
# saved. INTER_CALL_DELAY spaces out the 4 Groq-touching calls *within*
# one question; ROW_DELAY adds a further pause between questions. Both
# are deliberately generous -- slower but it actually finishes.
INTER_CALL_DELAY = 2
ROW_DELAY = 3

for suffix, table in variant_tables.items():
    print(f"=== Running config {suffix} ===")
    for _, row in questions_df.iterrows():
        try:
            is_unanswerable = row["type"] == "unanswerable"
            # Wrapped the same way as the feedback calls below -- retrieval
            # (Snowflake) plus the answer-generation Groq call have no
            # hard timeout of their own otherwise, and this is the one
            # call in the loop that was NOT covered by _with_retry before.
            aq_result = _with_retry(
                answer_question, table, row["question"], row["company"], row["fiscal_year"], is_unanswerable,
                max_attempts=1, call_timeout=90,
            )
            if aq_result is None:
                answer, sources, context_text = "ERROR: answer_question timed out or failed", [], ""
            else:
                answer, sources, context_text = aq_result

            # Cap context length fed to the feedback scorer -- groundedness
            # and context-relevance scoring are the most token-hungry calls
            # (full excerpt text as the "source"/"context" argument), and
            # trimming them cuts per-call token usage well under the TPM
            # cap without meaningfully changing what's being measured.
            feedback_context = context_text[:6000]
            groundedness = _with_retry(groundedness_score, feedback_context, answer)
            time.sleep(INTER_CALL_DELAY)
            context_relevance = _with_retry(context_relevance_score, row["question"], feedback_context)
            time.sleep(INTER_CALL_DELAY)
            answer_relevance = _with_retry(answer_relevance_score, row["question"], answer)
            time.sleep(INTER_CALL_DELAY)
            accuracy = _with_retry(judge_accuracy, row["question"], row["expected_answer"], answer)

            all_rows.append({
                "config": suffix,
                "question": row["question"],
                "type": row["type"],
                "company": row["company"],
                "fiscal_year": row["fiscal_year"],
                "expected_answer": row["expected_answer"],
                "answer": answer,
                "sources": ";".join(sources),
                "groundedness": groundedness,
                "context_relevance": context_relevance,
                "answer_relevance": answer_relevance,
                "accuracy": accuracy,
                "refused": is_refusal(answer),
            })
        except Exception as e:
            print(f"  [warn] row failed entirely, skipping: {type(e).__name__}: {e}")
            all_rows.append({
                "config": suffix, "question": row["question"], "type": row["type"],
                "company": row["company"], "fiscal_year": row["fiscal_year"],
                "expected_answer": row["expected_answer"], "answer": f"ERROR: {e}",
                "sources": "", "groundedness": None, "context_relevance": None,
                "answer_relevance": None, "accuracy": None, "refused": False,
            })

        done += 1
        # Save after every row, not just at the end -- so a crash we
        # didn't anticipate still leaves partial results on disk instead
        # of losing the whole run (which is exactly what happened the
        # first time: zero rows survived a mid-run crash).
        pd.DataFrame(all_rows).to_csv("eval/results/raw_results.csv", index=False)
        print(f"  [{done}/{total}] {suffix} :: {row['question'][:60]}...")
        time.sleep(ROW_DELAY)

results_df = pd.DataFrame(all_rows)
results_df.to_csv("eval/results/raw_results.csv", index=False)
print(f"Saved {len(results_df)} rows to eval/results/raw_results.csv")
results_df.head()


## 5. Compare configs and pick the best one

Aggregates per-config averages across groundedness, context relevance,
answer relevance, and LLM-judged accuracy, plus the refusal rate measured
only on the `unanswerable` rows (a well-grounded pipeline should refuse
those, so **higher** refusal rate there is better).

In [ ]:
quality_summary = (
    results_df
    .groupby("config")
    .agg(
        avg_groundedness=("groundedness", "mean"),
        avg_context_relevance=("context_relevance", "mean"),
        avg_answer_relevance=("answer_relevance", "mean"),
        accuracy=("accuracy", "mean"),  # mean() skips NaN automatically -> unanswerable rows excluded
    )
    .reset_index()
)

refusal_summary = (
    results_df[results_df["type"] == "unanswerable"]
    .groupby("config")["refused"]
    .mean()
    .rename("refusal_rate")
    .reset_index()
)

comparison_summary = quality_summary.merge(refusal_summary, on="config", how="left")
comparison_summary["composite_score"] = comparison_summary[
    ["avg_groundedness", "avg_context_relevance", "avg_answer_relevance", "accuracy", "refusal_rate"]
].mean(axis=1)

comparison_summary = comparison_summary.sort_values("composite_score", ascending=False).reset_index(drop=True)
comparison_summary.to_csv("eval/results/comparison_summary.csv", index=False)
comparison_summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
metrics = ["avg_groundedness", "avg_context_relevance", "avg_answer_relevance", "accuracy", "refusal_rate"]
comparison_summary.set_index("config")[metrics].plot(kind="bar", ax=ax)
ax.set_ylabel("Score (0-1)")
ax.set_title("FilingIQ chunking configuration comparison")
ax.set_ylim(0, 1)
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("eval/results/comparison_chart.png", dpi=150)
plt.show()


In [ ]:
best_row = comparison_summary.iloc[0]
print(f"Recommended chunking configuration: {best_row['config']}  (composite score: {best_row['composite_score']:.3f})")
print()
print(comparison_summary.to_string(index=False))
print()
print("Results written to eval/results/raw_results.csv, eval/results/comparison_summary.csv, eval/results/comparison_chart.png")


## 6. (Optional) clean up the evaluation variants

The production pipeline only needs `DOC_CHUNKS` (built by
`sql/01_setup.sql` + `sql/03_process_procedure.sql`). Once you've picked
a winning config and (if needed) re-pointed `PROCESS_NEW_DOCS()`'s default
`CHUNK_SIZE`/`CHUNK_OVERLAP` to match it, you can drop the three
evaluation variant tables.

In [ ]:
# Uncomment to drop the evaluation variants once you're done comparing.
# for table in variant_tables.values():
#     run_sql(f"DROP TABLE IF EXISTS {table}")
# print("Evaluation variants dropped.")

# conn.close()
